# Event-Episode v2 — Colab Runner

End-to-end pipeline for the v2 event-episode experiment.  Runs:

1. **Tabular HPO** (Phase 4) — LR/RF/XGB sweep over (T, L, W).
2. **DL HPO + multi-seed** (Phase 5) — GRU/LSTM/RNN/CNN.
3. **Sensor ablation** (Phase 6).
4. **Leakage probe** (Phase 7) — gate before final test.
5. **Final test evaluation** (Phase 8).

All outputs are written under `outputs/v2/tables/`.  This notebook does **not** modify any v1 file.

## 1. Clone / pull repo, install deps, mount Drive

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/bahaa1515/EECE-693-project.git"
REPO_DIR = Path("/content/EECE-693-project")
BRANCH = "codex/tarek-event-episode-tuning"

if REPO_DIR.exists():
    %cd /content/EECE-693-project
    !git fetch --all --quiet
    !git checkout {BRANCH}
    !git pull --ff-only
else:
    %cd /content
    !git clone {REPO_URL} EECE-693-project
    %cd /content/EECE-693-project
    !git checkout {BRANCH}

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Mount Drive and point at the AAMOS dataset / output folder.
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped:", exc)

data_dir = Path(os.environ.get("AAMOS_DATA_DIR", "/content/drive/MyDrive/AAMOS/dataset"))
out_dir  = Path(os.environ.get("AAMOS_OUTPUT_DIR", "/content/drive/MyDrive/AAMOS/outputs"))
if data_dir.exists():
    os.environ["AAMOS_DATA_DIR"] = str(data_dir)
    os.environ["AAMOS_OUTPUT_DIR"] = str(out_dir)
    print("AAMOS_DATA_DIR =", os.environ["AAMOS_DATA_DIR"])
    print("AAMOS_OUTPUT_DIR =", os.environ["AAMOS_OUTPUT_DIR"])
else:
    print("Set AAMOS_DATA_DIR to the Drive folder containing the AAMOS CSVs.")

## 2. Configure the experiment

Set the (T, L, W) grid here.  Smaller grids run faster on Colab CPUs;
the defaults match the plan.

In [ ]:
THRESHOLDS = "2,3,4"
LENGTHS    = "3,7,14"      # L=28 omitted for DL (plan); tabular tolerates it
WASHOUTS   = "0,7,14"
SEED       = 42
DL_EPOCHS  = 30
DL_BATCH   = 32
N_SHUFFLES = 5
MULTI_SEEDS = "42,43,44,45,46"

## 3. Phase 4 — Tabular HPO

In [ ]:
!python -m scripts.v2.tune_tabular_v2 \
    --thresholds {THRESHOLDS} \
    --lengths {LENGTHS} \
    --washouts {WASHOUTS} \
    --seed {SEED}

In [ ]:
import pandas as pd
best = pd.read_csv("outputs/v2/tables/tune_tabular_v2_best.csv")
best.sort_values("val_pr_auc", ascending=False).head(20)

## 4. Phase 5 — Deep-learning HPO + multi-seed

In [ ]:
!python -m scripts.v2.tune_dl_v2 \
    --thresholds {THRESHOLDS} \
    --lengths {LENGTHS} \
    --washouts {WASHOUTS} \
    --seed {SEED} \
    --multi-seeds {MULTI_SEEDS} \
    --epochs {DL_EPOCHS} \
    --batch-size {DL_BATCH}

In [ ]:
ms = pd.read_csv("outputs/v2/tables/tune_dl_v2_multiseed.csv")
ms

## 5. Phase 6 — Sensor ablation

In [ ]:
!python -m scripts.v2.sensor_ablation_v2

In [ ]:
abl = pd.read_csv("outputs/v2/tables/sensor_ablation_v2.csv")
abl.sort_values("val_pr_auc", ascending=False).head(30)

## 6. Phase 7 — Leakage probe (gate before Phase 8)

If any winner's shuffled-label ROC-AUC falls outside `[0.45, 0.55]`,
this prints a WARNING and you should investigate before trusting the
Phase-8 numbers.

In [ ]:
!python -m scripts.v2.leakage_probe_v2 --n-shuffles {N_SHUFFLES}

## 7. Phase 8 — Final test-set evaluation

In [ ]:
!python -m scripts.v2.final_test_eval_v2 \
    --dl-epochs {DL_EPOCHS} \
    --dl-batch-size {DL_BATCH}

In [ ]:
summary = pd.read_csv("outputs/v2/tables/final_test_v2_summary.csv")
summary

## 8. (Optional) Copy v2 outputs back to Drive

Useful when Colab's local FS will be wiped at session end.

In [ ]:
import shutil
drive_target = Path("/content/drive/MyDrive/AAMOS/outputs/v2")
if drive_target.parent.exists():
    drive_target.mkdir(parents=True, exist_ok=True)
    shutil.copytree("outputs/v2", drive_target, dirs_exist_ok=True)
    print("copied -> ", drive_target)
else:
    print("Drive output folder not present; skipping copy.")